# Retrieval-Augmented Agent



A retrieval-augmented agent fetches relevant knowledge before it answers or acts.



## Stack

- Framework: LangChain prompt pipeline with local retrieval

- LLM: local Ollama via `ChatOllama(model="llama3.1:latest")`

- Pattern goal: grounded responses



## When to use it

- The answer depends on private or changing knowledge

- You want grounding instead of pure model recall

## Architecture



```mermaid

graph LR

    Q[Question] --> RET[Retriever]

    RET --> CTX[Relevant Context]

    CTX --> L[Local Ollama Prompt]

    Q --> L

    L --> A[Grounded Answer]

```

In [ ]:
from visual_diagram_helper import render_svg_flow



nodes = [

    ("q", "Question", 60, 170, "#bfdbfe"),

    ("ret", "Retriever", 290, 170, "#bbf7d0"),

    ("ctx", "Top Context", 520, 170, "#fde68a"),

    ("llm", "LLM Prompt", 750, 170, "#fecaca"),

    ("ans", "Grounded Answer", 980, 170, "#ddd6fe"),

]

edges = [

    ("q", "ret"),

    ("ret", "ctx"),

    ("ctx", "llm"),

    ("llm", "ans"),

]



render_svg_flow("RAG Visual Architecture", nodes, edges, width=1220, height=340)

In [2]:
from agent_patterns_common import (

    append_history,

    get_local_llm,

    invoke_with_retry,

    log_event,

)





knowledge_base = [

    "The router pattern selects the right workflow for a request.",

    "The planner-executor pattern separates planning from doing.",

    "Memory helps an agent preserve user preferences across interactions.",

    "Reflection loops improve output quality by adding self-critique.",

]





def retrieve_context(question: str) -> list[str]:

    question_words = set(question.lower().split())

    scored = []

    for item in knowledge_base:

        overlap = len(question_words.intersection(item.lower().split()))

        if overlap:

            scored.append((overlap, item))

    ranked = [item for _, item in sorted(scored, reverse=True)]

    log_event("info", "retrieval_complete", match_count=len(ranked), question=question)

    return ranked[:2]





llm = get_local_llm()

history = []

question = "How does the router workflow help?"

context_items = retrieve_context(question)

history = append_history(history, f"Retrieved context: {context_items}")

prompt = (

    "Answer using only provided context.\n"

    "Return EXACTLY one sentence with <= 20 words.\n"

    f"Context:\n{chr(10).join(context_items)}\n\n"

    f"History:\n{history}\n\n"

    f"Question:\n{question}"

)



answer = invoke_with_retry(llm, prompt)

{

    "question": question,

    "context_items": context_items,

    "history": history,

    "answer": answer,

}

{"level": "INFO", "event_type": "retrieval_complete", "match_count": 2, "question": "How does the router workflow help?"}
{"level": "INFO", "event_type": "llm_invoke_start", "attempt": 1, "prompt_preview": "Answer using only provided context.\nReturn EXACTLY one sentence with <= 20 words"}
{"level": "INFO", "event_type": "llm_invoke_success", "attempt": 1}


{'question': 'How does the router workflow help?',
 'context_items': ['The router pattern selects the right workflow for a request.',
  'The planner-executor pattern separates planning from doing.'],
 'history': ["Retrieved context: ['The router pattern selects the right workflow for a request.', 'The planner-executor pattern separates planning from doing.']"],
 'answer': 'The router pattern helps by selecting the correct workflow for each incoming request.'}

## Design insight



Retrieval narrows the model's working set to the relevant facts. That is what makes answers more grounded and easier to trust.